# Cloud Storage (GCS) Hands-On Lab — Python-Orchestrated Google Cloud SDK
### GCP Training Program — Day 1, Module 1: Storage & Ingestion

**What this notebook covers:** the exact same Cloud Storage operations as the other two GCS labs —
buckets, uploads/downloads, listing, copy/rename/delete, metadata, storage classes, lifecycle rules,
versioning, ACLs & IAM, signed URLs, CORS, and object composition — but this time by calling the
**`gcloud storage`** CLI *from Python* via `subprocess`, parsing its JSON output, and acting on the
result programmatically.

**How this differs from the other two labs:**
- The **Python SDK lab** talks to GCS directly through `google-cloud-storage` — no CLI involved.
- The **CLI lab** runs `gcloud storage` commands directly in notebook cells using `!` — great for
  copy-paste into shell scripts, but each cell is a standalone, human-read command.
- **This lab** is the pattern you'd actually use to build a real automation tool, a Composer/Airflow
  operator, or a CI/CD step in Python: shell out to `gcloud` with `subprocess`, capture structured
  JSON (`--format=json`), and let Python make decisions based on that output — for example, finding
  the oldest object generation automatically instead of a human reading it off the screen and pasting
  it back in.

**This notebook is fully self-contained.** Just run the cells top to bottom:
1. The **Authenticate** cell will pop up a Google sign-in flow.
2. The **Configure** cell will *ask you* for your Project ID, region, and bucket name.
3. Every operation after that is a Python function call that shells out to `gcloud` under the hood.

**Prerequisites (mention to the class before running):**
1. A GCP project with billing enabled.
2. IAM role on the project: `roles/storage.admin` (or equivalent) for the account you sign in with.
3. The Cloud SDK (`gcloud`) — already pre-installed in Colab, nothing to install.


## 1. Authenticate This Session
Sign in with the Google account that has access to your GCP project. In Colab this also configures
the `gcloud` CLI that Python will be shelling out to for the rest of this notebook.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated in Colab. This also configures the gcloud CLI for this session.")
else:
    print("Not running in Colab — assuming gcloud Application Default Credentials are set up.")
    print("If this is your first time, run in a terminal:")
    print("  gcloud auth login")
    print("  gcloud auth application-default login")

## 2. Build a Reusable `run_gcloud()` Helper
This is the core pattern for this notebook: a single Python function that shells out to `gcloud`,
raises a clear error on failure instead of silently continuing, and — when asked — parses `--format=json`
output straight into a Python `dict`/`list` you can act on. Every section below calls this same helper.

In [ ]:
import subprocess
import json

def run_gcloud(args, input_text=None, as_json=False, check=True):
    """Run a gcloud/gcloud storage command from Python and optionally parse JSON output.

    Args:
        args: list of CLI args, e.g. ["storage", "buckets", "list", "--format=json"]
        input_text: optional string piped to stdin (used for `cp -` style uploads)
        as_json: if True, parse stdout as JSON and return it (add --format=json to args yourself)
        check: if True, raise RuntimeError on a non-zero exit code
    """
    cmd = ["gcloud"] + args
    result = subprocess.run(cmd, input=input_text, capture_output=True, text=True)

    if check and result.returncode != 0:
        print("STDERR:\n", result.stderr)
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(cmd)}")

    if as_json:
        return json.loads(result.stdout) if result.stdout.strip() else None

    if result.stdout.strip():
        print(result.stdout)
    return result.stdout

# Sanity check
run_gcloud(["version"])

## 3. Configure Your Project, Region & Bucket
Enter your own values when prompted below — nothing to hand-edit in code. The bucket name gets a
timestamp suffix so it's globally unique (bucket names are global across all of GCS).

In [ ]:
import time

PROJECT_ID = input("Enter your GCP Project ID: ").strip()

REGION = input("Enter your region [default: asia-south1]: ").strip()
if not REGION:
    REGION = "asia-south1"

_default_bucket = f"gcs-pysdk-training-demo-{int(time.time())}"
BUCKET_NAME = input(f"Enter a bucket name [default: {_default_bucket}]: ").strip()
if not BUCKET_NAME:
    BUCKET_NAME = _default_bucket

run_gcloud(["config", "set", "project", PROJECT_ID])
print("Project set to:", PROJECT_ID)
print("Region:", REGION)
print("Target bucket:", BUCKET_NAME)

## 4. Bucket Operations
### 4.1 Create a Bucket

In [ ]:
run_gcloud([
    "storage", "buckets", "create", f"gs://{BUCKET_NAME}",
    "--project", PROJECT_ID,
    "--location", REGION,
    "--default-storage-class=STANDARD",
])
print(f"Created gs://{BUCKET_NAME}")

### 4.2 List All Buckets in the Project
Parsed as JSON straight into a Python list, so we can act on the names — not just read them off
the screen.

In [ ]:
buckets = run_gcloud(["storage", "buckets", "list", "--project", PROJECT_ID, "--format=json"], as_json=True)

bucket_names = [b["name"] for b in buckets]
print(f"{len(bucket_names)} bucket(s) in project {PROJECT_ID}:")
for name in bucket_names:
    print(" -", name)

assert BUCKET_NAME in bucket_names, "Newly created bucket not found in listing yet — check for propagation delay."

### 4.3 Get Bucket Metadata

In [ ]:
bucket_info = run_gcloud(["storage", "buckets", "describe", f"gs://{BUCKET_NAME}", "--format=json"], as_json=True)

print("Name:          ", bucket_info["name"])
print("Location:      ", bucket_info["location"])
print("Storage class: ", bucket_info["storageClass"])
print("Created:       ", bucket_info["timeCreated"])

### 4.4 Update Bucket Configuration (labels + uniform access)

In [ ]:
run_gcloud([
    "storage", "buckets", "update", f"gs://{BUCKET_NAME}",
    "--update-labels=env=training,module=gcs-lab",
    "--uniform-bucket-level-access",
])

bucket_info = run_gcloud(["storage", "buckets", "describe", f"gs://{BUCKET_NAME}", "--format=json"], as_json=True)
assert bucket_info["labels"]["env"] == "training"
print("Labels applied:", bucket_info["labels"])
print("Uniform bucket-level access:", bucket_info["iamConfiguration"]["uniformBucketLevelAccess"])

## 5. Object Upload / Download
### 5.1 Upload a Local File
Create the local file with plain Python `open()` — no shell needed even for that part.

In [ ]:
with open("sample.txt", "w") as f:
    f.write("Hello from the GCP Cloud Storage hands-on lab (Python + gcloud SDK edition)!\n")

run_gcloud(["storage", "cp", "sample.txt", f"gs://{BUCKET_NAME}/uploads/sample.txt"])
print(f"Uploaded to gs://{BUCKET_NAME}/uploads/sample.txt")

### 5.2 Upload From In-Memory Data (no local file needed)
`input_text` pipes a Python string straight into `gcloud storage cp -` via stdin — the CLI's
equivalent of the Python client's `upload_from_string`, but driven entirely from Python.

In [ ]:
import json as _json  # local alias just for this cell's payload construction

payload = _json.dumps({"lab": "gcs", "day": 1})
run_gcloud(["storage", "cp", "-", f"gs://{BUCKET_NAME}/uploads/inmemory.json"], input_text=payload)
print("Uploaded in-memory object: uploads/inmemory.json")

### 5.3 Download an Object to a Local File

In [ ]:
run_gcloud(["storage", "cp", f"gs://{BUCKET_NAME}/uploads/sample.txt", "sample_downloaded.txt"])

with open("sample_downloaded.txt") as f:
    print("Downloaded content ->", f.read())

### 5.4 Download an Object Directly Into a Python Variable
`gcloud storage cat` writes to stdout, which `subprocess` captures directly into a Python string —
no local file touched at all.

In [ ]:
content = run_gcloud(["storage", "cat", f"gs://{BUCKET_NAME}/uploads/inmemory.json"])
print("In-memory content:", content.strip())

## 6. Listing & Organizing Objects
### 6.1 List All Objects in the Bucket — With a Python-Computed Total Size

In [ ]:
objects = run_gcloud(["storage", "objects", "list", f"gs://{BUCKET_NAME}", "--format=json"], as_json=True)

total_size = sum(int(o.get("size", 0)) for o in objects)
print(f"{len(objects)} object(s), {total_size} bytes total:")
for o in objects:
    print(" -", o["name"], "-", o.get("size", "?"), "bytes")

### 6.2 List Objects Under a Prefix (simulated "folder")

In [ ]:
objects = run_gcloud([
    "storage", "objects", "list", f"gs://{BUCKET_NAME}/uploads/", "--format=json",
], as_json=True)

for o in objects:
    print(o["name"])

### 6.3 Folder-Style (Delimited) Listing
`--delimiter=/` mirrors the Python client's `delimiter` parameter, returning one level at a time
instead of a fully recursive listing.

In [ ]:
result = run_gcloud([
    "storage", "objects", "list", f"gs://{BUCKET_NAME}/", "--delimiter=/", "--format=json",
], as_json=True)

for o in result:
    print(o.get("name") or o.get("prefix"))

## 7. Copy, Rename, Move & Delete Objects
### 7.1 Copy an Object

In [ ]:
run_gcloud([
    "storage", "cp",
    f"gs://{BUCKET_NAME}/uploads/sample.txt",
    f"gs://{BUCKET_NAME}/archive/sample_copy.txt",
])
print("Copied to archive/sample_copy.txt")

### 7.2 Rename / Move an Object
GCS has no native rename — `mv` copies to the new name then deletes the original, same as the
Python client's `rename_blob`.

In [ ]:
run_gcloud([
    "storage", "mv",
    f"gs://{BUCKET_NAME}/archive/sample_copy.txt",
    f"gs://{BUCKET_NAME}/archive/sample_renamed.txt",
])
print("Renamed/moved to archive/sample_renamed.txt")

### 7.3 Delete an Object

In [ ]:
run_gcloud(["storage", "rm", f"gs://{BUCKET_NAME}/archive/sample_renamed.txt"])
print("Deleted archive/sample_renamed.txt")

## 8. Object Metadata
### 8.1 Read System Metadata

In [ ]:
obj_info = run_gcloud([
    "storage", "objects", "describe", f"gs://{BUCKET_NAME}/uploads/sample.txt", "--format=json",
], as_json=True)

print("Size:        ", obj_info["size"])
print("Content-Type:", obj_info.get("contentType"))
print("Generation:  ", obj_info["generation"])

### 8.2 Set Custom Metadata & Content-Type, Then Verify Programmatically

In [ ]:
run_gcloud([
    "storage", "objects", "update", f"gs://{BUCKET_NAME}/uploads/sample.txt",
    "--custom-metadata=lab-owner=instructor,purpose=gcs-demo",
    "--content-type=text/plain",
])

obj_info = run_gcloud([
    "storage", "objects", "describe", f"gs://{BUCKET_NAME}/uploads/sample.txt", "--format=json",
], as_json=True)

assert obj_info["metadata"]["lab-owner"] == "instructor"
print("Custom metadata confirmed:", obj_info["metadata"])

## 9. Storage Classes
| Class | Min. storage duration | Typical access pattern | Relative cost |
|---|---|---|---|
| STANDARD | none | frequent access | highest storage, lowest access |
| NEARLINE | 30 days | ~monthly access | lower storage, small access fee |
| COLDLINE | 90 days | ~quarterly access | lower still, higher access fee |
| ARCHIVE | 365 days | rare/backup | lowest storage, highest access fee |

**Cost warning:** changing storage class or deleting objects before the minimum storage duration
incurs an early-deletion charge. Don't run this against production data.

### 9.1 Change the Bucket's Default Storage Class

In [ ]:
run_gcloud(["storage", "buckets", "update", f"gs://{BUCKET_NAME}", "--default-storage-class=NEARLINE"])

bucket_info = run_gcloud(["storage", "buckets", "describe", f"gs://{BUCKET_NAME}", "--format=json"], as_json=True)
print("Bucket default storage class is now:", bucket_info["storageClass"])

### 9.2 Change the Storage Class of an Existing Object

In [ ]:
run_gcloud(["storage", "objects", "update", f"gs://{BUCKET_NAME}/uploads/sample.txt", "--storage-class=COLDLINE"])

obj_info = run_gcloud([
    "storage", "objects", "describe", f"gs://{BUCKET_NAME}/uploads/sample.txt", "--format=json",
], as_json=True)
print("Object storage class is now:", obj_info["storageClass"])

## 10. Object Lifecycle Management
### 10.1 Define & Apply Lifecycle Rules
Build the rule as a Python `dict` and write it with `json.dump` — no hand-written JSON file, and no
risk of a typo breaking the parse.

In [ ]:
lifecycle_rules = {
    "rule": [
        {"action": {"type": "SetStorageClass", "storageClass": "COLDLINE"}, "condition": {"age": 30}},
        {"action": {"type": "Delete"}, "condition": {"age": 365}},
    ]
}

with open("lifecycle.json", "w") as f:
    json.dump(lifecycle_rules, f, indent=2)

run_gcloud(["storage", "buckets", "update", f"gs://{BUCKET_NAME}", "--lifecycle-file=lifecycle.json"])
print("Lifecycle rules applied.")

### 10.2 View the Current Lifecycle Configuration

In [ ]:
bucket_info = run_gcloud(["storage", "buckets", "describe", f"gs://{BUCKET_NAME}", "--format=json"], as_json=True)
for rule in bucket_info.get("lifecycle", {}).get("rule", []):
    print(rule)

### 10.3 Remove All Lifecycle Rules
**Caveat:** the exact flag for clearing lifecycle rules has moved around across `gcloud` releases.
The fallback below (writing an empty rule list and re-applying) works regardless of SDK version.

In [ ]:
try:
    run_gcloud(["storage", "buckets", "update", f"gs://{BUCKET_NAME}", "--clear-lifecycle"])
except RuntimeError:
    print("--clear-lifecycle not supported on this SDK version, falling back to an empty rule file.")
    with open("lifecycle_empty.json", "w") as f:
        json.dump({"rule": []}, f)
    run_gcloud(["storage", "buckets", "update", f"gs://{BUCKET_NAME}", "--lifecycle-file=lifecycle_empty.json"])

bucket_info = run_gcloud(["storage", "buckets", "describe", f"gs://{BUCKET_NAME}", "--format=json"], as_json=True)
print("Lifecycle rules after removal:", bucket_info.get("lifecycle", {}).get("rule", []))

## 11. Object Versioning
### 11.1 Enable Versioning

In [ ]:
run_gcloud(["storage", "buckets", "update", f"gs://{BUCKET_NAME}", "--versioning"])

bucket_info = run_gcloud(["storage", "buckets", "describe", f"gs://{BUCKET_NAME}", "--format=json"], as_json=True)
print("Versioning enabled:", bucket_info["versioning"]["enabled"])

### 11.2 Create Multiple Versions & List Generations
A Python loop drives three uploads to the same object name — each becomes a new generation instead
of overwriting in place.

In [ ]:
for i in range(1, 4):
    run_gcloud(
        ["storage", "cp", "-", f"gs://{BUCKET_NAME}/uploads/versioned.txt"],
        input_text=f"version {i}",
    )

versions = run_gcloud([
    "storage", "objects", "list", f"gs://{BUCKET_NAME}/uploads/versioned.txt",
    "--all-versions", "--format=json",
], as_json=True)

print("All generations of uploads/versioned.txt:")
for v in versions:
    print(" generation:", v["generation"], "| size:", v.get("size"), "bytes")

### 11.3 Delete a Specific Version (generation)
This is the payoff of the programmatic approach: Python sorts the parsed generations and deletes the
**oldest one automatically** — no manual copy-pasting a generation number off the screen, unlike the
pure CLI version of this lab.

In [ ]:
oldest = sorted(versions, key=lambda v: int(v["generation"]))[0]

run_gcloud(["storage", "rm", f"gs://{BUCKET_NAME}/uploads/versioned.txt#{oldest['generation']}"])
print("Deleted generation:", oldest["generation"])

## 12. Access Control — ACLs & IAM
**Teaching point:** fine-grained ACLs and uniform bucket-level access (IAM-only) are mutually
exclusive. We enabled uniform access in Section 4.4, so we disable it here to demo legacy ACLs, then
show the modern IAM-based approach.
### 12.1 Disable Uniform Bucket-Level Access

In [ ]:
run_gcloud(["storage", "buckets", "update", f"gs://{BUCKET_NAME}", "--no-uniform-bucket-level-access"])
print("Uniform bucket-level access disabled - fine-grained ACLs are now usable.")

### 12.2 Grant Read Access to a Specific User via ACL

In [ ]:
run_gcloud([
    "storage", "objects", "update", f"gs://{BUCKET_NAME}/uploads/sample.txt",
    "--add-acl-grant=entity=user-someone@example.com,role=READER",
])
print("Granted READ on uploads/sample.txt to someone@example.com")

### 12.3 Make an Object Publicly Readable
**Use with caution in a live demo** — this exposes the object to the entire internet.

In [ ]:
run_gcloud([
    "storage", "objects", "update", f"gs://{BUCKET_NAME}/uploads/sample.txt",
    "--add-acl-grant=entity=allUsers,role=READER",
])
print("Public URL:", f"https://storage.googleapis.com/{BUCKET_NAME}/uploads/sample.txt")

### 12.4 IAM Policy at the Bucket Level (recommended over ACLs)

In [ ]:
run_gcloud([
    "storage", "buckets", "add-iam-policy-binding", f"gs://{BUCKET_NAME}",
    "--member=group:data-readers@example.com",
    "--role=roles/storage.objectViewer",
])
print("IAM policy updated on bucket:", BUCKET_NAME)

### 12.5 View the Current Bucket IAM Policy

In [ ]:
policy = run_gcloud(["storage", "buckets", "get-iam-policy", f"gs://{BUCKET_NAME}", "--format=json"], as_json=True)

for binding in policy.get("bindings", []):
    print(binding["role"], "->", binding["members"])

## 13. Signed URLs — Temporary, Credential-less Access
**Note for class:** `gcloud storage sign-url` requires either a service account private key file or
`--impersonate-service-account` with `roles/iam.serviceAccountTokenCreator` on that service account —
plain end-user credentials usually can't sign directly. Skip live execution if the class doesn't have
a service account handy; walk through the pattern instead.
### 13.1 Signed URL for Downloading (GET)

In [ ]:
SERVICE_ACCOUNT_EMAIL = "your-service-account@your-project.iam.gserviceaccount.com"  # <-- replace

signed_url = run_gcloud([
    "storage", "sign-url", f"gs://{BUCKET_NAME}/uploads/sample.txt",
    "--duration=15m",
    "--http-verb=GET",
    f"--impersonate-service-account={SERVICE_ACCOUNT_EMAIL}",
])
print("Signed GET URL (valid 15 min):\n", signed_url)

### 13.2 Signed URL for Uploading (PUT)

In [ ]:
signed_put_url = run_gcloud([
    "storage", "sign-url", f"gs://{BUCKET_NAME}/uploads/via_signed_url.txt",
    "--duration=15m",
    "--http-verb=PUT",
    f"--impersonate-service-account={SERVICE_ACCOUNT_EMAIL}",
])
print("Signed PUT URL (valid 15 min):\n", signed_put_url)

## 14. CORS Configuration
Needed when a browser-based app on a different origin needs to call GCS directly (e.g. direct-to-
bucket uploads). Same pattern as lifecycle: build the config as a Python `dict`/`list`, write it with
`json.dump`, then apply it.

In [ ]:
cors_config = [
    {
        "origin": ["https://example.com"],
        "method": ["GET", "HEAD"],
        "responseHeader": ["Content-Type"],
        "maxAgeSeconds": 3600,
    }
]

with open("cors.json", "w") as f:
    json.dump(cors_config, f, indent=2)

run_gcloud(["storage", "buckets", "update", f"gs://{BUCKET_NAME}", "--cors-file=cors.json"])

bucket_info = run_gcloud(["storage", "buckets", "describe", f"gs://{BUCKET_NAME}", "--format=json"], as_json=True)
print("CORS configuration applied:", bucket_info.get("cors"))

## 15. Compose Objects (server-side concatenation)
Combines multiple objects into one **without downloading and re-uploading** — useful after parallel
chunked uploads.

In [ ]:
run_gcloud(["storage", "cp", "-", f"gs://{BUCKET_NAME}/compose/part1.txt"], input_text="Hello, ")
run_gcloud(["storage", "cp", "-", f"gs://{BUCKET_NAME}/compose/part2.txt"], input_text="World!")

run_gcloud([
    "storage", "objects", "compose",
    f"gs://{BUCKET_NAME}/compose/part1.txt",
    f"gs://{BUCKET_NAME}/compose/part2.txt",
    f"gs://{BUCKET_NAME}/compose/combined.txt",
])

combined_content = run_gcloud(["storage", "cat", f"gs://{BUCKET_NAME}/compose/combined.txt"])
print("Composed object content:", combined_content)

## 16. Cleanup — Delete All Objects & the Bucket
Run this **last**, once every demo above has been shown, so the class sees teardown too. Python lists
every live object *and* every archived version, then deletes each one before removing the bucket
itself — the same logic a real cleanup script would use.

In [ ]:
all_versions = run_gcloud([
    "storage", "objects", "list", f"gs://{BUCKET_NAME}", "--all-versions", "--format=json",
], as_json=True)

for obj in all_versions:
    run_gcloud(["storage", "rm", f"gs://{BUCKET_NAME}/{obj['name']}#{obj['generation']}"], check=False)

run_gcloud(["storage", "buckets", "delete", f"gs://{BUCKET_NAME}"])
print(f"Bucket {BUCKET_NAME} and all contents deleted.")